# LMIP Silver Layer - Complete Documentation

The Silver layer is the **core transformation and quality control hub** of the Labour Market Intelligence Platform. It transforms raw bronze data into clean, standardized, validated, and enriched job postings ready for semantic analysis and warehousing.

---

## Architecture Principles

* **Idempotency**: All notebooks can be re-run safely without duplicating data
* **Parameterization**: Widget-based parameters for Databricks Workflows integration
* **Deterministic-first**: Rule-based logic before ML/statistical fallbacks
* **Audit trail**: All changes logged to `silver_job_changes`
* **Graceful degradation**: Low-confidence items queued for review rather than rejected

---

## Pipeline Flow

```
┌─────────────────────────────────────────────────────────────────┐
│                    BRONZE LAYER (Raw APIs)                      │
│                  bronze.bronze_job_snapshot                     │
└────────────────────┬────────────────────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────────────┐
│  1. STANDARDIZE JOBS (silver_standardize_jobs)                  │
│     • Normalize company names, titles, locations                │
│     • Map source-specific fields to standard schema             │
│     • Generate record hashes for change detection               │
│     ➜ Output: silver_jobs_staging                               │
└────────────────────┬────────────────────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────────────┐
│  2. DETECT CDC (silver_detect_cdc)                              │
│     • Compare staging vs current (INSERT/UPDATE/DELETE)         │
│     • Merge changes to silver_jobs_current                      │
│     • Log all changes to silver_job_changes                     │
│     ➜ Output: silver_jobs_current (updated)                     │
└────────────────────┬────────────────────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────────────┐
│  3. DATA QUALITY VALIDATION (silver_dq_validate)                │
│     • Completeness, uniqueness, referential integrity           │
│     • Format validation (URLs, enums)                           │
│     • Business rules (title length, salary ranges)              │
│     • FAIL ➜ Quarantine, WARN ➜ Flag, PASS ➜ Proceed           │
│     ➜ Output: DQ flags in current table                         │
└────────────────────┬────────────────────────────────────────────┘
                     │
        ┌────────────┴────────────┐
        ▼                         ▼
┌─────────────────────┐  ┌─────────────────────┐
│  4a. SKILL EXTRACT  │  │  4b. SECTOR ASSIGN  │
│  (silver_skill_     │  │  (silver_sector_    │
│   extract)          │  │   assign)           │
│  • Keyword matching │  │  • Company patterns │
│  • Pattern regex    │  │  • Title keywords   │
│  • Evidence capture │  │  • Confidence score │
│  ➜ silver_skill_    │  │  ➜ Sector fields in │
│     mapping         │  │     current table   │
└─────────────────────┘  └─────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────────────────────┐
│  5. JOB IDENTITY MAPPING (silver_job_identity_map)              │
│     • Cross-source duplicate detection                          │
│     • Exact match, fuzzy company, composite hash                │
│     • Link jobs to enterprise_job_id                            │
│     ➜ Output: silver_job_identity_map                           │
└─────────────────────────────────────────────────────────────────┘

                SUPPORTING NOTEBOOKS
┌─────────────────────────────────────────────────────────────────┐
│  • SOFT DELETE/RESTORE (silver_apply_soft_delete_restore)       │
│    ➜ Logical deletion without physical removal                  │
│  • SEMANTIC REVIEW QUEUE (silver_semantic_review_queue)         │
│    ➜ Manual review workflow for low-confidence items            │
│  • QUARANTINE ROUTING (silver_quarantine_route)                 │
│    ➜ Manage DQ-failed records, reinstate or purge              │
└─────────────────────────────────────────────────────────────────┘
```

---

## Notebook Reference

### 1. silver_standardize_jobs

**Purpose**: Transform raw bronze API responses into standardized silver schema

**Key Operations**:
- Company name normalization (uppercase, remove legal suffixes)
- Job title normalization
- Location parsing and normalization
- Remote type detection (REMOTE, HYBRID, ONSITE)
- Generate record hash for CDC (company + title + description + location)

**Parameters**:
- `batch_id`: Specific batch to process (default: latest)
- `source_filter`: Comma-separated source names
- `mode`: incremental (default) or full

**Outputs**:
- `silver.silver_jobs_staging`

**Idempotency**: Overwrites staging table per batch

---

### 2. silver_detect_cdc

**Purpose**: Detect changes (CDC) between staging and current silver tables

**Key Operations**:
- **INSERT**: New jobs in staging not in current
- **UPDATE**: Jobs with changed record_hash
- **DELETE**: Jobs missing from staging (soft delete with `is_active=False`)
- **RESTORE**: Deleted jobs reappearing

**Parameters**:
- `batch_id`: Batch to process
- `enable_deletes`: true (default) or false

**Outputs**:
- `silver.silver_jobs_current` (merged)
- `silver.silver_job_changes` (audit log)

**Idempotency**: Delta merge is idempotent

---

### 3. silver_dq_validate

**Purpose**: Apply data quality rules and route failures

**Quality Checks**:
1. **Completeness**: Required fields not null
2. **Uniqueness**: enterprise_job_id is unique
3. **Referential integrity**: Valid source_name
4. **Format validation**: URL patterns, enum values
5. **Business rules**: Title length, company not empty

**Parameters**:
- `validation_level`: strict, standard (default), permissive
- `quarantine_on_fail`: true (default) or false

**Outputs**:
- `dq_*` columns in `silver_jobs_current`
- `quarantine.quarantine_jobs` (failed records)

**Idempotency**: DQ flags updated via merge

---

### 4. silver_skill_extract

**Purpose**: Extract skills from job descriptions and titles

**Extraction Methods**:
1. **Keyword matching**: 80+ skill keywords (python, react, aws, etc.)
2. **Pattern matching**: Regex for skill mentions
3. **Evidence capture**: 60-char text snippet with skill mention

**Parameters**:
- `extraction_method`: all (default), keyword, pattern, nlp
- `batch_limit`: Max records to process

**Outputs**:
- `silver.silver_skill_mapping`

**Skill Categories**:
- Programming languages (Python, Java, JavaScript, etc.)
- Frameworks (React, Django, Spring, etc.)
- Cloud (AWS, Azure, GCP)
- Data (Spark, Pandas, TensorFlow)
- DevOps (Docker, Kubernetes, Jenkins)

**Idempotency**: Only processes jobs not yet in skill_mapping

---

### 5. silver_sector_assign

**Purpose**: Assign industry sector using hierarchical rules

**Assignment Strategy** (in priority order):
1. **Company pattern** (conf: 0.85): "tech", "software", "bank", "pharma"
2. **Title keywords** (conf: 0.75): "developer", "accountant", "nurse"
3. **Default**: "unknown" (conf: 0.0)

**Sectors**:
- Technology, Finance, Healthcare, Retail, Manufacturing, Education, Consulting, Media, Energy, Transportation

**Parameters**:
- `confidence_threshold`: Minimum confidence for auto-assignment (default: 0.7)
- `queue_low_confidence`: true (default) - queue for review if below threshold

**Outputs**:
- Sector fields in `silver_jobs_current`
- `silver_semantic_review_queue` (low confidence)

**Idempotency**: Updates existing sector assignments

---

### 6. silver_job_identity_map

**Purpose**: Detect duplicate jobs across different sources

**Matching Rules** (hierarchical):
1. **Exact match** (score: 1.0): Same company + title + location
2. **Fuzzy company** (score: 0.85): Similar company + same title
3. **Composite hash** (score: 0.80): Hash of company + title + description

**Parameters**:
- `match_threshold`: Minimum score (default: 0.75)
- `create_new_enterprise_ids`: true (default) for unmatched jobs

**Outputs**:
- `silver.silver_job_identity_map`

**Use Case**: Enable cross-source analytics ("job X appears on 3 platforms")

**Idempotency**: Appends new matches only

---

### 7. silver_apply_soft_delete_restore

**Purpose**: Manage logical deletion lifecycle

**Operations**:
- **delete**: Set `soft_delete_flag=True`, `is_active=False`
- **restore**: Clear soft delete, set `is_active=True`

**Parameters**:
- `operation`: delete or restore
- `job_ids`: Comma-separated enterprise_job_ids
- `reason`: Reason for operation
- `source_filter`: Limit to specific source

**Outputs**:
- Updated `silver_jobs_current`
- Logged to `silver_job_changes`

**Idempotency**: Delta merge handles re-runs

---

### 8. silver_semantic_review_queue

**Purpose**: Manage manual review workflow

**Issue Types**:
- SECTOR_LOW_CONFIDENCE
- TITLE_UNCLEAR
- SKILL_AMBIGUOUS
- COMPANY_UNKNOWN
- LOCATION_INVALID

**Actions**:
- `list_pending`: View items awaiting review
- `resolve`: Mark item as resolved with notes
- `dismiss`: Mark as not an issue
- `summary`: Analytics on review queue

**Parameters**:
- `action`: list_pending (default), resolve, dismiss, summary
- `review_id`: ID for resolve/dismiss
- `resolution_notes`: Notes for resolution
- `issue_type_filter`: Filter by issue type

**Outputs**:
- Updates `silver.silver_semantic_review_queue`

**Workflow States**: PENDING → IN_REVIEW → RESOLVED/DISMISSED

---

### 9. silver_quarantine_route

**Purpose**: Manage quarantined records

**Actions**:
- `list`: View quarantined records
- `reinstate`: Move back to active (clears soft_delete_flag)
- `purge`: Permanently remove from quarantine
- `summary`: Analytics on quarantine

**Parameters**:
- `action`: list (default), reinstate, purge, summary
- `enterprise_job_ids`: IDs for reinstate/purge
- `reason_filter`: Filter by quarantine reason

**Outputs**:
- Updates `quarantine.quarantine_jobs`
- Logs to `silver.silver_job_changes`

**Integration**: Quarantined jobs excluded from downstream processing

---

## Silver Schema Tables

### Core Tables

#### silver.silver_jobs_current
**Purpose**: Current state snapshot of all job postings

**Key Fields**:
- `enterprise_job_id` (PK): Unique job identifier
- `source_name`, `source_job_id`, `source_job_key`: Source tracking
- `company_name_raw`, `company_name_norm`: Company fields
- `title_raw`, `title_normalized`: Job title
- `description_raw`: Full description
- `location_raw`, `location_norm`: Location
- `remote_type`: REMOTE, HYBRID, ONSITE
- `is_active`, `soft_delete_flag`: Status flags
- `record_hash`: Change detection fingerprint
- `dq_overall_status`: PASS, WARN, FAIL
- `sector_assigned`, `sector_confidence`: Sector fields
- `created_at`, `updated_at`: Timestamps

---

#### silver.silver_job_changes
**Purpose**: Audit trail of all job changes

**Key Fields**:
- `change_id` (PK): Unique change event
- `enterprise_job_id`: FK to current
- `source_name`: Source system
- `change_type`: INSERT, UPDATE, DELETE, RESTORE, SOFT_DELETE
- `changed_columns`: Array of column names that changed
- `old_value_json`: Previous values as JSON string
- `new_value_json`: New values as JSON string
- `change_timestamp`: When change occurred
- `batch_id`: Batch that detected change

---

#### silver.silver_skill_mapping
**Purpose**: Skills extracted from job postings

**Key Fields**:
- `skill_mapping_id` (PK): Unique mapping
- `enterprise_job_id`: FK to current
- `skill_name_raw`: Skill as found in text
- `skill_name_normalized`: Canonical skill name
- `extraction_method`: KEYWORD, PATTERN, NLP, LLM
- `evidence_text`: Text snippet with skill mention
- `confidence`: 0.0 - 1.0
- `extracted_at`: Timestamp

---

#### silver.silver_job_identity_map
**Purpose**: Cross-source duplicate detection

**Key Fields**:
- `job_identity_map_sk` (PK): Surrogate key
- `job_id_1`, `job_id_2`: Matched job IDs
- `source_1`, `source_2`: Source names
- `match_rule`: EXACT_MATCH, FUZZY_COMPANY, COMPOSITE_HASH
- `match_score`: Similarity score
- `assigned_at`: Timestamp

---

### Supporting Tables

#### quarantine.quarantine_jobs
**Purpose**: DQ-failed records awaiting remediation

**Key Fields**:
- `quarantine_id` (PK): Unique quarantine record
- `enterprise_job_id`: Job in quarantine
- `source_name`, `source_job_id`: Source tracking
- `failure_stage`: Stage where failure occurred
- `failed_rule_name`: Specific rule that failed
- `severity`: CRITICAL, HIGH, MEDIUM, LOW
- `payload_json`: Full record as JSON
- `quarantined_at`: Timestamp
- `reprocess_status`: PENDING, RESOLVED, DISCARDED
- `reprocess_batch_id`, `resolved_at`: Reprocessing tracking

---

#### silver.silver_semantic_review_queue
**Purpose**: Items requiring manual semantic review

**Key Fields**:
- `review_id` (PK): Unique review item
- `enterprise_job_id`: Job to review
- `issue_type`: SECTOR_AMBIGUOUS, TITLE_UNCLEAR, etc.
- `issue_detail`: Description
- `confidence`: How confident the issue is real
- `status`: PENDING, IN_REVIEW, RESOLVED, DISMISSED
- `queued_at`, `resolved_at`: Timestamps
- `resolution_notes`: Resolution details

---

#### silver.silver_jobs_staging
**Purpose**: Temporary staging for standardized records before CDC

**Usage**: Written by `nb_silver_standardize_jobs`, consumed by `nb_silver_detect_cdc`

---

## Execution Patterns

### Daily Incremental Load

```python
# Databricks Workflow orchestration

# Task 1: Standardize latest batch
dbutils.notebook.run(
    "/LMIP/notebooks/silver/silver_standardize_jobs",
    timeout_seconds=3600,
    arguments={"mode": "incremental"}
)

# Task 2: Detect CDC
dbutils.notebook.run(
    "/LMIP/notebooks/silver/silver_detect_cdc",
    timeout_seconds=3600,
    arguments={"enable_deletes": "true"}
)

# Task 3: Data quality validation
dbutils.notebook.run(
    "/LMIP/notebooks/silver/silver_dq_validate",
    timeout_seconds=3600,
    arguments={"validation_level": "standard", "quarantine_on_fail": "true"}
)

# Task 4a: Extract skills (parallel)
dbutils.notebook.run(
    "/LMIP/notebooks/silver/silver_skill_extract",
    timeout_seconds=3600,
    arguments={"extraction_method": "all"}
)

# Task 4b: Assign sectors (parallel)
dbutils.notebook.run(
    "/LMIP/notebooks/silver/silver_sector_assign",
    timeout_seconds=3600,
    arguments={"confidence_threshold": "0.7", "queue_low_confidence": "true"}
)

# Task 5: Identity mapping
dbutils.notebook.run(
    "/LMIP/notebooks/silver/silver_job_identity_map",
    timeout_seconds=3600,
    arguments={"match_threshold": "0.75"}
)
```

---

### Full Reprocessing

When schema changes or logic updates require full reprocessing:

```python
# 1. Truncate current table
spark.sql("TRUNCATE TABLE silver.silver_jobs_current")

# 2. Run standardize + CDC for ALL batches
dbutils.notebook.run(
    "/LMIP/notebooks/silver/silver_standardize_jobs",
    timeout_seconds=7200,
    arguments={"mode": "full", "batch_id": ""}  # Empty = all batches
)

dbutils.notebook.run(
    "/LMIP/notebooks/silver/silver_detect_cdc",
    timeout_seconds=7200,
    arguments={"enable_deletes": "false"}  # No deletes on full load
)

# 3. Continue with DQ, skills, sectors, identity mapping
```

---

### Manual Review Workflow

```python
# 1. List pending reviews
dbutils.notebook.run(
    "/LMIP/notebooks/silver/silver_semantic_review_queue",
    timeout_seconds=300,
    arguments={"action": "list_pending", "issue_type_filter": "SECTOR_LOW_CONFIDENCE"}
)

# 2. Resolve specific review
dbutils.notebook.run(
    "/LMIP/notebooks/silver/silver_semantic_review_queue",
    timeout_seconds=300,
    arguments={
        "action": "resolve",
        "review_id": "<review_id>",
        "resolution_notes": "Sector manually assigned to Technology"
    }
)
```

---

### Quarantine Management

**Note**: Quarantine operations are managed in the dedicated **Quarantine layer**, not Silver.

See `/LMIP/notebooks/quarantine/README_QUARANTINE` for complete documentation.

```python
# 1. List quarantined records
dbutils.notebook.run(
    "/LMIP/notebooks/quarantine/quarantine_manage",
    timeout_seconds=300,
    arguments={"action": "list", "reason_filter": "DQ_FAILURE"}
)

# 2. Reinstate after correction
dbutils.notebook.run(
    "/LMIP/notebooks/quarantine/quarantine_manage",
    timeout_seconds=300,
    arguments={
        "action": "reinstate",
        "enterprise_job_ids": "<job_id_1>,<job_id_2>"
    }
)

# 3. View quarantine summary
dbutils.notebook.run(
    "/LMIP/notebooks/quarantine/quarantine_manage",
    timeout_seconds=300,
    arguments={"action": "summary"}
)
```

---

## Best Practices & Guidelines

### Performance Optimization

1. **Batch Size**: Process 1000-10000 records per batch for optimal performance
2. **Partitioning**: Use `batch_id` and `source_name` for partition pruning
3. **Caching**: Avoid caching; Delta tables are already optimized
4. **Parallel Processing**: Run skill extraction and sector assignment in parallel

---

### Error Handling

1. **Graceful Degradation**: Low-confidence items queue for review instead of failing
2. **Quarantine Strategy**: DQ failures quarantined, not dropped
3. **Audit Trail**: All operations logged to `silver_job_changes`
4. **Idempotency**: All notebooks can be re-run without duplicating data

---

### Data Quality

1. **Validation Levels**:
   - **Strict**: Fail on any DQ issue (use for critical pipelines)
   - **Standard**: Warn on minor issues, fail on critical (default)
   - **Permissive**: Log all issues, fail only on schema violations

2. **Confidence Thresholds**:
   - **High confidence** (≥0.85): Auto-process
   - **Medium confidence** (0.70-0.84): Auto-process with review flag
   - **Low confidence** (<0.70): Queue for manual review

---

### Monitoring

**Key Metrics to Track**:
1. CDC change ratios (inserts:updates:deletes)
2. DQ pass rate (target: >95%)
3. Quarantine growth rate
4. Skill extraction coverage (% jobs with skills)
5. Sector assignment confidence distribution
6. Cross-source duplicate rate
7. Semantic review queue size

**Alerting**:
- DQ pass rate drops below 90%
- Quarantine exceeds 5% of total jobs
- Review queue exceeds 100 items
- Zero records processed in a batch (upstream issue)

---

### Extension Points

1. **New Skills**: Add to `SKILL_KEYWORDS` dict in `nb_silver_skill_extract`
2. **New Sectors**: Add to `COMPANY_SECTOR_PATTERNS` in `nb_silver_sector_assign`
3. **Custom DQ Rules**: Add checks to `nb_silver_dq_validate`
4. **New Matching Rules**: Add to `nb_silver_job_identity_map`
5. **LLM Integration**: Replace deterministic logic with LLM calls (future)

---

### Troubleshooting

**Issue**: No records in staging
- **Check**: Bronze ingestion ran successfully
- **Check**: `batch_id` parameter is correct
- **Fix**: Run bronze ingestion first

**Issue**: CDC detects no changes
- **Check**: `record_hash` logic is stable
- **Check**: Staging table has data
- **Fix**: Verify hash generation includes all significant fields

**Issue**: DQ failures spike
- **Check**: Source data quality degraded
- **Check**: New source added without schema mapping
- **Fix**: Update standardization logic or adjust DQ rules

**Issue**: Skill extraction returns 0 skills
- **Check**: Description field is populated
- **Check**: `SKILL_KEYWORDS` dict loaded
- **Fix**: Verify UDF registration and test with sample data

---

## Next Steps: Integration with Semantic Layer

The silver layer outputs feed into the **semantic layer** for advanced enrichment:

1. **Role Canonicalization** (`semantic.sem_job_role_map`)
   - Map job titles to standard role taxonomy (e.g., "Senior Python Dev" → "Software Engineer - Senior")
   - Uses silver skill data for disambiguation

2. **Company Canonicalization** (`semantic.sem_company_canonical`)
   - Resolve company name variations ("Google Inc" → "Google")
   - Add company metadata (size, industry, location)

3. **Skills Taxonomy** (`semantic.sem_skill_taxonomy`)
   - Normalize extracted skills to canonical taxonomy
   - Build skill hierarchies (Python → Programming → Technical Skills)

4. **Location Enrichment** (`semantic.sem_location_map`)
   - Parse raw locations to structured fields (city, state, country)
   - Add geo-coordinates, timezone, metro area

5. **Salary Normalization** (`semantic.sem_salary_norm`)
   - Convert salary ranges to standard currency/timeframe
   - Adjust for cost-of-living, experience level

---

## Contact & Support

**Silver Layer Owner**: Data Engineering Team  
**Documentation**: `/LMIP/docs/silver_layer.md`  
**Issues**: File in project Jira under "LMIP - Silver Layer"  
**Slack**: #lmip-data-engineering

---

## Version History

**v1.0** (2026-06-04)
- Initial silver layer implementation
- 9 notebooks covering standardization, CDC, DQ, enrichment
- Supports Arbeitnow and Remotive sources
- Keyword-based skill extraction (80+ skills)
- Rule-based sector assignment (10 sectors)
- 3-tier matching for cross-source identity

**Planned Enhancements**:
- LLM-based skill extraction (Q3 2026)
- ML-based sector classification (Q4 2026)
- Real-time streaming CDC (2027)
- Auto-remediation for common DQ issues (2027)